<a href="https://colab.research.google.com/github/KaisaridiSofia/leaspy_tutorial/blob/fit_notebook/notebooks/fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# On Colab the tutorial's own files (tutolib/, content/) are cloned from GitHub.
# BRANCH picks which version students get; set it to "main" once this work is
# merged there. To re-pull after changing it, delete the leaspy_tutorial folder.
BRANCH = "main"
REPO = "https://github.com/aramis-lab/leaspy_tutorial"

import importlib.util, pathlib, subprocess, sys

try:  # find_spec raises (not returns None) when there is no `google` package at all
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
pip = [sys.executable, "-m", "pip", "install", "-q"]

if IN_COLAB:
    if importlib.util.find_spec("leaspy") is None:
        subprocess.run(pip + ["--no-deps",
    "leaspy @ git+https://github.com/aramis-lab/leaspy.git@v2.1.0"], check=True)
        subprocess.run(pip + ["lifelines"], check=True)   # dep Colab may lack
    if not pathlib.Path("leaspy_tutorial").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO], check=True)
    ROOT = pathlib.Path("leaspy_tutorial")
else:
    # running from a clone: walk up from the notebook until we find the repo root
    ROOT = next(p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents)
                if (p / "tutolib").is_dir())
sys.path.insert(0, str(ROOT))

import tutolib as tp
print("tutolib", tp.__version__, "loaded from", ROOT)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import leaspy
from leaspy.datasets import load_dataset
from leaspy.models import LogisticModel
from leaspy.io.logs.visualization.plotting import Plotting

print(f"leaspy {leaspy.__version__}")



# Data

Leaspy is a library for analyzing **longitudinal data**: repeated observations of the same subjects over time. To use it, your data needs to be loaded in a specific format.

Leaspy expects data as a **pandas DataFrame** with:
- a `MultiIndex` of `(ID, TIME)`, where `ID` identifies the subject and `TIME` is the age (or time) at each visit
- one column per **feature**

Additionally, for the logistic model, each feature must be:
- **increasing with disease severity** (a higher value = more progressed disease)
- **normalized between 0 and 1**

Real-world data rarely satisfies this out of the box — some scores decrease with severity, others aren't on a `[0, 1]` scale. Let's look at an example.

In [ ]:
df_raw = pd.read_csv(tp.asset_path("data_raw.csv"), index_col=["ID", "TIME"])
df_raw.head()

In [ ]:
# Check the range of each feature
df_raw.agg(["min", "max"])

In [ ]:
# Check whether each feature tends to increase or decrease with TIME, on average
corr = df_raw.reset_index("TIME").groupby("ID").apply(lambda x: x.corr(numeric_only=True)["TIME"]).mean().drop("TIME").rename_axis(None)
pd.DataFrame({
    "correlation": corr,
    "trend": np.where(corr > 0, "increasing", "decreasing"),
})

`Feature_1` ranges from 0 to 100 and is positively correlated with time — it just needs to be rescaled to `[0, 1]`. `Feature_2` is already in `[0, 1]`, but negatively correlated with time — it decreases with disease severity, so we'll need to invert it. Let's fix both.

In [ ]:
df_clean = df_raw.copy()

### Normalize a feature

`Feature_1` ranges between 0 and 100 and is already increasing with disease severity. Rescale it to `[0, 1]` in df_clean.

In [ ]:
tp.solution(1)

### Invert a decreasing feature

`Feature_2` ranges between 0 and 1 but *decreases* with disease progression. Transform it in df_clean so it increases with severity.

In [ ]:
tp.solution(2)

### Keep subjects with at least 2 visits

Leaspy needs repeated observations per subject to estimate a trajectory, so we only keep subjects seen at least twice.

In [ ]:
# Check for patients with only one visit
n_visits = df_clean.groupby("ID").size()
n_single_visit = (n_visits == 1).sum()
print(f"{n_single_visit} out of {n_visits.shape[0]} patients have only one visit.")

Now, keep only the subjects with **at least 2 visits**.

In [ ]:
tp.solution(3)

## From here on: the Parkinson dataset

Now that you've seen how to bring raw data into the right format, we'll use a dataset that is **already preprocessed**: Leaspy's built-in synthetic Parkinson's disease dataset, with 200 subjects and three normalized clinical scores.

In [ ]:
df = load_dataset("parkinson")
FEATURES = ["MDS1_total", "SCOPA_total", "MOCA_total"]
SEED = 0
df = df[FEATURES]
print(df.head())

n_subjects = df.index.get_level_values("ID").nunique()
print(f"{n_subjects} subjects in the dataset.")

avg_visits = df.groupby("ID").size().mean()
print(f"Average number of visits per subject: {avg_visits:.1f}")

### Train / test split

As with any predictive model, we hold out a subset of subjects as a **test set**, kept aside from model fitting. This lets us later check how well the model generalizes to subjects it hasn't seen during training.

In [ ]:
# Train / test split
df_train = df.loc[:"GS-160"]
df_test = df.loc["GS-161":]
train_ids = df_train.index.unique("ID")
test_ids = df_test.index.unique("ID")
print(f"Train : {len(train_ids)} subjects")
print(f"Test  : {len(test_ids)} subjects")

In [ ]:
df_train.head()

### Spaghetti plots

A good first step before fitting any model is to visualize the raw trajectories. 
Using `df_train`, plot the trajectories of each feature (one color per feature) as a function of age, with one line per subject (**spaghetti plot** of the data).

In [ ]:
tp.solution(4)

# Fit

Once we have a dataset that is ready for leaspy, we have to choose which model to fit in our data.

For now the available models are:
- `LogisticModel`
- `LinearModel`
- `JointModel`
- `LogisticMultivariateMixtureModel`

For this example we will run a logistic model. Since we have three scores we will run a multivariate model wo se need to choose beforehand the number of sources.

*Reminder:* The sources represent the groups of scores that evolve in a similar way. It should be a number smaller that the total number of features.

In [ ]:
tp.runquestion(1)

In [ ]:
# Load the model
model_2_sources = LogisticModel(name="logistic", source_dimension=2)

In [ ]:
# Fit the model
MODEL_2_LOGS = Path("_outputs") / "model_2_sources"

model_2_sources.fit(
    df_train, "mcmc_saem",
    seed=SEED, n_iter=1000, progress_bar=True,
    save_periodicity=50,
    path=MODEL_2_LOGS,
    overwrite_logs_folder=True,
)

In [ ]:
# Save the model
model_2_sources.save(MODEL_2_LOGS / "model_parameters.json")

## Convergence assessment

The model is estimated via an MCMC-SAEM. It is an iterative algorithm that lies in a stochastic approximation. Before interpreting the results of our fit we should check if the model has converged. In simple words we should see if the final estimations are stable and if the chains had explored adequately the parameter space.

During the fit, the value of every estimated parameter is recorded each `save_periodicity` iterations into a `.csv` file under `_outputs/model_2_sources/parameter_convergence/`. Those files are what we plot below.

Read the traces from left to right: early on they drift as the algorithm searches, then they should settle into a flat band that only jitters. The dashed line marks the end of the **burn-in** phase, after which leaspy starts averaging, so it is the flatness *to the right of it* that tells you the fit has converged.

In [ ]:
PARAMS = ["tau_mean", "tau_std", "xi_std", "noise_std", "g", "v0"]

def plot_convergence(folder, params=PARAMS, features=FEATURES, burn_in=None):
    """Plot one MCMC trace per parameter, straight from the CSVs leaspy saved.
    """
    fig, axes = plt.subplots(3, 2, figsize=(11, 9))
    for ax, name in zip(axes.flat, params):
        trace = pd.read_csv(Path(folder) / f"{name}.csv", index_col=0, header=None)
        if trace.shape[1] == len(features):
            trace.columns = features  # one curve per score, so label them
        trace.plot(ax=ax, lw=1, legend=False)
        end_of_burn_in = burn_in if burn_in is not None else 0.9 * trace.index.max()
        ax.axvline(end_of_burn_in, ls="--", c="0.5")  # end of burn-in
        ax.set_title(name)
        ax.set_xlabel("")

    handles = [plt.Line2D([], [], color=f"C{i}") for i in range(len(features))]
    fig.legend(handles, features, loc="upper center", ncol=len(features), frameon=False)
    fig.supxlabel("iteration")
    fig.tight_layout(rect=(0, 0, 1, 0.96))  # leave a strip at the top for the legend

In [ ]:
# Your own fit wrote one CSV per parameter into MODEL_2_LOGS/parameter_convergence/.
plot_convergence(MODEL_2_LOGS / "parameter_convergence")
plt.show()

### Question

Take a look at the convergence plots above before answering.

In [ ]:
tp.runquestion(2)

In [ ]:
# The canonical 100 000-iteration run shipped with the tutorial
RUN = ROOT / "content" / "reference_run"

plot_convergence(RUN / "parameter_convergence")
plt.show()

**Note:** as shown above, 1000 iterations are not enough for the model to fully converge, so the parameters you just fitted are not the ones you would report in a real analysis.

Rather than draw conclusions from an under-converged fit, we ship the **converged** model: the result of that same 100 000-iteration run, saved to a JSON file. Loading it is instant, so from here on we can look at trustworthy parameters and trajectories without anyone waiting 11 minutes.

In [ ]:
# Restoring a saved fit is a one-liner: same class, same API as the model you fitted above
model_100k = LogisticModel.load(str(RUN / "parkinson_fit_V21.json"))

print(type(model_100k).__name__, model_100k.features, f"source_dimension={model_100k.source_dimension}")
# on a loaded model these two are plain dicts, so they are read with [] not a dot
print(f"trained for {model_100k.training_info['n_iter']} iterations "
      f"on {model_100k.dataset_info['n_subjects']} subjects")

## Model summary

Now that we have a model with a satisfactory convergence, we can look at its fit output via the integrated `summary()` function. Everything from here on reports `model_100k`, the converged fit -- not the 1000-iteration one you trained above.

In [ ]:
model_100k.summary()

We can show specific elements such as the metrics:

In [ ]:
summary = model_100k.summary()
print(f"BIC: {float(summary.bic)}")
print(f"AIC: {float(summary.aic)}")

Or a specific set of parameters

In [ ]:
model_100k.summary().parameters['Individual Parameters']

## Plotting

We can plot the population trajectories of the converged model with leaspy's internal plotting tool.

In [ ]:
ax = Plotting(model_100k).average_trajectory(
    alpha=1, figsize=(14, 6), n_std_left=2, n_std_right=8
)
plt.show()

In [ ]:
tp.show_asset("sigmoid_interactive.html")

## Model Comparison

The exercises below use `model_2_sources`, the 1,000-iteration fit, together with other deliberately short fits.

AIC and BIC should be compared on the same observations only after every candidate model has converged. Because the 1,000-iteration fits in this section are intentionally under-converged, the exercise demonstrates the comparison API but its ranking should not be used for scientific model selection. A real comparison would fit every candidate until its traces stabilize, preferably across multiple seeds.

### Task 1: Vary the number of sources

Can you load and fit a logistic model with one source and compare it with the previous model?

model_1_source = ...

In [ ]:
# If you need help, run this cell to get the code
# Click `Show Solution` to see the answer
tp.solution(5)

In [ ]:
# Now compare the two models with their BIC / AIC
tp.solution(6)

In [ ]:
tp.runquestion(3)

### Task 2: Vary the noise model

The default choice is to fit a `gaussian-diagonal` noise, which means that we fit a noise with a separate standard deviation per score


In [ ]:
model_2_sources.observation_model_names

Another option is to fit a `gaussian-scalar` noise, which means fitting one global standard deviation for the noise shared between all the scores.

We just have to add the argument `obs_models='gaussian-scalar'` when we load our model.

In [ ]:
tp.runquestion(4)

In [ ]:
# Fit a 1-source model with a scalar noise
tp.solution(7)

Let's fit also a model with 2 sources and a scalar noise and compare the four models all together.

In [ ]:
# Fit a 2-source model with a scalar noise
tp.solution(8)

In [ ]:
# Compare the four models with their BIC / AIC
tp.solution(9)